In [ ]:
import pandas as pd

In [ ]:
latest_df = pd.read_csv("./pipeline_steps/input_files/combined_classified_data.csv")

In [ ]:
latest_df["source_file"].value_counts()

In [ ]:
latest_df.columns

In [ ]:
pra_1 = pd.read_csv(
    "./pipeline_steps/input_files/classified/PRA_DME_Cases_10-2023_to_7-2024__classified.csv"
)
ucla_rm = pd.read_csv(
    "./pipeline_steps/input_files/classified/UCLA_Romero_Ruby_Request_10.10.24_classified.csv"
)
ucla_aa = pd.read_csv(
    "./pipeline_steps/input_files/classified/UCLAAllClosedCases_classified.csv"
)

In [ ]:
case_nums_1 = set(pra_1["CaseNum"])
case_nums_2 = set(ucla_rm["CaseNum"])
case_nums_3 = set(ucla_aa["CaseNum"])

case_nums_1.update(case_nums_2)

In [ ]:
diff = case_nums_1.difference(case_nums_3)

In [ ]:
latest_df.drop(columns=["DeathTime"], inplace=True)

In [ ]:
duplicates = latest_df[latest_df.duplicated(subset="CaseNumber", keep=False)]
print(duplicates.head())  # Displays the first few duplicate rows

In [ ]:
duplicates_sorted = duplicates.sort_values(by="CaseNumber")

In [ ]:
grouped = duplicates_sorted.groupby("CaseNumber")


# Step 4: Define a function to compare rows within each group
def find_differences(group):
    # If the group has only one row, no differences to find
    if len(group) == 1:
        return pd.DataFrame()
    else:
        # Initialize an empty DataFrame to store differences
        differences = pd.DataFrame()
        # Get all column names except 'CaseNumber'
        columns = group.columns.drop("CaseNumber")
        # Compare each column
        for column in columns:
            # Check if all values in the column are the same
            if group[column].nunique() > 1:
                # If not, include this column in differences
                differences[column] = group[column]
        # Add 'CaseNumber' to keep track of the group
        differences["CaseNumber"] = group["CaseNumber"]
        return differences


# Step 5: Apply the function to each group and collect the results
differences_list = []
for name, group in grouped:
    diff = find_differences(group)
    if not diff.empty:
        differences_list.append(diff)

# Concatenate all differences into a single DataFrame
differences_df = pd.concat(differences_list)

In [ ]:
pd.set_option("display.max_columns", None)

In [ ]:
differences_df.to_csv("./pipeline_differences.csv")

In [ ]:
# Assume df is your DataFrame
# List of columns to check
columns_to_check = [
    "Methamphetamine",
    "Heroin",
    "Cocaine",
    "Fentanyl",
    "Alcohol",
    "Prescription.opioids",
    "Any Opioids",
    "Benzodiazepines",
    "Others",
    "Any Drugs",
    "Drug No Opioids",
    "EventAddress",
]


# Function to resolve duplicates based on your criteria
def resolve_duplicates(group):
    # If there's only one row, return it as is
    if len(group) == 1:
        return group

    # Step 1: Keep rows where the specified columns have '1'
    for col in columns_to_check[:-1]:  # Exclude 'EventAddress' for now
        max_value = group[col].max()
        group = group[group[col] == max_value]
        if len(group) == 1:
            return group

    # Step 2: If the values are the same, compare 'EventAddress' length
    group["EventAddress_length"] = group["EventAddress"].astype(str).str.len()
    max_length = group["EventAddress_length"].max()
    group = group[group["EventAddress_length"] == max_length]
    group = group.drop(columns="EventAddress_length")

    # Step 3: If still multiple rows, keep the first one (it doesn't matter which)
    return group.iloc[[0]]


# Apply the function to each group of duplicates
processed_df = latest_df.groupby("CaseNumber", group_keys=False).apply(
    resolve_duplicates
)

# Reset index if needed
processed_df = processed_df.reset_index(drop=True)

# Display the resulting DataFrame
print(processed_df)

In [ ]:
processed_df.to_csv("2021-01-2024-06-ods.csv")